# ResNet

In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

### ResNet 34

In [2]:
from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Layer, Dense, Dropout, Flatten, Conv2D, MaxPool2D
from tensorflow.keras.layers import BatchNormalization, Activation, GlobalAvgPool2D

In [4]:
# residual 잔차 전달이 진행되는 layer
class Residual(Layer):    
    def __init__ (self, filters, strides = 1, activation = 'relu', **kwargs):
        super(). __init__(**kwargs)
        self.activation = keras.activations.get(activation)
        self.main_layers = [
                    Conv2D(filters, 3, strides = strides,
                                        padding = 'same', use_bias = False),
                    BatchNormalization(),
                    self.activation,
                    Conv2D(filters, 3, strides = 1,
                                        padding = 'same', use_bias = False),
                    BatchNormalization()
        ]
        
        self.skip_layers = []
        if strides > 1: 
            #strides가 2일 경우 차원을 맞춰줘야함
            self.skip_layers = [
                    Conv2D(filters, 1, strides = strides,
                            padding = 'same', use_bias = False),
                    BatchNormalization()]
    
    # input값을 layer의 최종 output과 더해주는 함수
    def __call__(self,inputs):
        z = inputs # 이전 layer의 output값
        
        for layer in self.main_layers:
            z = layer(z)
            # 이전 layer의 output값을 적용한 main layer 결과값
            
        # identity mapping : 최초의 input값을 저장    
        skip_z = inputs
        
        # skip_layer에 값이 있는 경우 = 이전 filter 수와 달라지는 경우
        # 차원을 갖게 만들어주는 연산 수행 결과를 skip_z에 더해줌
        # projection shortcut
        for layer in self.skip_layers:
            skip_z = layer(skip_z)
        
        # skip_layer에 값이 없으면 그냥 inputs값이 더해진다. = identity shortcut
        # ReLU를 적용한 결과값 return
        return self.activation(z + skip_z)

![img](https://miro.medium.com/max/1400/1*aqmUx_ONo8KqKNEYsjM8eA.png)

![img](https://d2l.ai/_images/resnet-block.svg)

In [5]:
model = Sequential()
model.add(Conv2D(64, 7, strides = 2, input_shape = [224,224,3],
                  padding='same', use_bias = False))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPool2D(pool_size = 3, strides = 2, padding='same'))

prev_filters = 64
for filters in [64] * 3 + [128] * 4 + [256] * 6 + [512] * 3:
    strides = 1 if filters == prev_filters else 2 # 이전 필터 수와 같으면 1
    model.add(Residual(filters, strides = strides))
    prev_filters = filters
    
model.add(GlobalAvgPool2D())
model.add(Flatten())
model.add(Dense(1000, activation = 'softmax'))
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 112, 112, 64)      9408      
_________________________________________________________________
batch_normalization (BatchNo (None, 112, 112, 64)      256       
_________________________________________________________________
activation (Activation)      (None, 112, 112, 64)      0         
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 56, 56, 64)        0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 56, 56, 64)        36864     
_________________________________________________________________
batch_normalization_1 (Batch (None, 56, 56, 64)        256       
_________________________________________________________________
tf.nn.relu (TFOpLambda)      (None, 56, 56, 64)        0